In [ ]:
# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝

import requests
import pandas as pd
import numpy as np
from time import sleep
import os

# CONFIG ────
OUTPUT_DIR = "international_commodity_standardized_onebyone"
os.makedirs(OUTPUT_DIR, exist_ok=True)


UNIT_TO_SINGLE = {

    'Kg':                        1.0,
    '1 kg':                      1.0,
    '1.1 Kg':                    1.1,
    '1.5 Kg':                    1.5,
    '2 Kg':                      2.0,
    '2 kg':                      2.0,
    '2.5 kg':                    2.5,
    '3 kg':                      3.0,
    '3.5 kg':                    3.5,
    '5 kg':                      5.0,
    '10 Kg':                     10.0,
    '10 kg':                     10.0,
    '12.5 Kg':                   12.5,
    '20 kg':                     20.0,
    '25 kg':                     25.0,
    '30 kg':                     30.0,
    '50 kg':                     50.0,
    '60 kg':                     60.0,
    '100 Kg':                    100.0,
    '100 kg':                    100.0,
    'tonne':                     1000.0,
    '120 kg':                    120.0,
    '160 kg':                    160.0,
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.


. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.

. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.


. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.
.
.
.
. 
.
. 
.  "Full implementation removed for privacy"
. 
. 
.
.
.
.
.
.
.
.


    if not rows:
        return None

    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])
    df["date"] = df["date"].dt.to_period("M").dt.to_timestamp()

    df = (df.groupby("date", as_index=False)
            .apply(lambda g: g.loc[g.isna().sum(axis=1).idxmin()], include_groups=False)
            .reset_index(drop=True))

    full_range = pd.date_range(start=df["date"].min(),
                               end=df["date"].max(), freq="MS")
    df = (df.set_index("date")
            .reindex(full_range)
            .rename_axis("date")
            .reset_index())

    for col in ["commodity_name", "iso3_country_code", "country", "market",
                "price_type", "unit", "unit_std", "currency",
                "price_source", "fill_method"]:
        df[col] = df[col].ffill().bfill()

    df["fill_method"] = df["fill_method"].fillna("original")
    return df



# INTERNATIONAL SERIES

print("=" * 60)
print("FETCHING INTERNATIONAL SERIES")
print("=" * 60)

url_intl  = "https://fpma.fao.org/giews/v4/global/price_module/api/v1/FpmaSerieInternational/?limit=500"
intl_data = requests.get(url_intl).json()

commodity_buckets: dict[str, list[pd.DataFrame]] = {}

for item in intl_data["results"]:
    uuid           = item["uuid"]
    commodity_name = item.get("commodity_name", "Unknown")
    country        = item.get("market_name", "Unknown")
    iso3           = item.get("iso3_country_code", "Unknown")
    market         = item.get("country_name", "Unknown")
    price_type     = item.get("price_type", "Unknown")
    unit           = item.get("measure_unit_label", "Unknown")

    url_price = (
        f"https://fpma.fao.org/giews/v4/global/price_module/api/v1/"
        f"FpmaSeriePrice/?uuid__in={uuid}&periodicity=monthly"
    )
    resp = requests.get(url_price).json()
    if resp["count"] == 0:
        sleep(0.2)
        continue

    datapoints = resp["results"][0]["datapoints"]
    meta = dict(commodity_name=commodity_name, iso3=iso3, country=country,
                market=market, price_type=price_type, unit=unit, currency="USD")

    df = build_series_df(datapoints, meta, price_source="International")
    if df is None:
        sleep(0.2)
        continue

    mask = df["price_usd"].isna() & df["price_local"].notna()
    df.loc[mask, "price_usd"]   = df.loc[mask, "price_local"]
    df.loc[mask, "fill_method"] = "local_is_usd"

    df = recompute_price_per_unit(df)

    commodity_buckets.setdefault(commodity_name, []).append(df[FINAL_COLS])
    print(f"  {commodity_name} — {len(df)} rows")
    sleep(0.2)


#SAVE

print("\n" + "=" * 60)
print("SAVING — one CSV per commodity")
print("=" * 60)

import warnings

saved_count = 0
for commodity_name, frames in commodity_buckets.items():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        combined = pd.concat(frames, ignore_index=True)

    combined = combined.sort_values(["country", "market", "date"]).reset_index(drop=True)

    filename = f"{sanitize_filename(commodity_name)}.csv"
    filepath = os.path.join(OUTPUT_DIR, filename)
    combined.to_csv(filepath, index=False)
    saved_count += 1
    print(f"  Saved: {filepath}  ({len(combined):,} rows)")

print(f"\nTotal commodities saved : {saved_count}")
print(f"Output directory        : {OUTPUT_DIR}/")

# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝